# Ensemble (fixed)

Blends the tree models with the champion N-BEATS. Key fixes vs. the earlier version:

- Trees are retrained with the corrected preprocessing (Store/Dept features), so they
  are no longer stuck at ~14k.
- The neural component is the **real N-BEATS** predictions (from
  `predictions/nn_val_preds.csv`), not a placeholder.
- Weights are tuned honestly on the validation set (no rolling-window leakage).

All scored with the shared `calculate_wmae` on the same validation rows.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.preprocessing import merge_and_enrich, get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Ensemble_Fixed")

SPLIT_DATE = "2012-01-01"
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/11 20:52:09 INFO mlflow.tracking.fluent: Experiment with name 'Ensemble_Fixed' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date=SPLIT_DATE
)
print("train rows:", len(y_train), "| val rows:", len(y_val))

train rows: 294132 | val rows: 127438


## Train the trees (fixed preprocessing) and get validation predictions

In [4]:
with open("configs/xgboost_config.yaml") as f:
    xgb_params = yaml.safe_load(f)["model"]["params"]
with open("configs/lightgbm_config.yaml") as f:
    lgb_params = yaml.safe_load(f)["model"]["params"]

xgb_pipe = create_xgboost_pipeline(preprocessor, xgb_params)
xgb_pipe.fit(X_train, y_train)
xgb_val = xgb_pipe.predict(X_val)

lgb_pipe = create_lightgbm_pipeline(preprocessor, lgb_params)
lgb_pipe.fit(X_train, y_train)
lgb_val = lgb_pipe.predict(X_val)

print("XGBoost val WMAE:", round(calculate_wmae(y_val, xgb_val, is_holiday_val), 2))
print("LightGBM val WMAE:", round(calculate_wmae(y_val, lgb_val, is_holiday_val), 2))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012564 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2342
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 19
[LightGBM] [Info] Start training from score 16105.306894
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
XGBoost val WMAE: 3327.24
LightGBM val WMAE: 2824.63


## Align the N-BEATS validation predictions to the same rows

Rebuild the validation keys (Store, Dept, Date) in the same order as `y_val`, then
merge the exported N-BEATS predictions by key.

In [5]:
dfm = merge_and_enrich(train_raw, stores, features)
dfm = dfm.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
val_keys = dfm[dfm["Date"] >= pd.Timestamp(SPLIT_DATE)][["Store", "Dept", "Date"]].reset_index(drop=True)

nn = pd.read_csv("predictions/nn_val_preds.csv")
nn["Date"] = pd.to_datetime(nn["Date"])

nn_aligned = val_keys.merge(nn, on=["Store", "Dept", "Date"], how="left")
nn_val = nn_aligned["nn_pred"].fillna(0).values

print("aligned rows:", len(nn_val), "of", len(y_val))
print("N-BEATS val WMAE:", round(calculate_wmae(y_val, nn_val, is_holiday_val), 2))

aligned rows: 127438 of 127438
N-BEATS val WMAE: 2274.69


## Tune the blend weights honestly on validation

In [6]:
with mlflow.start_run(run_name="Ensemble_Weight_Search"):
    grid = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]
    best_wmae, best_w = float("inf"), None

    for w_nn in grid:
        for w_xgb in grid:
            w_lgb = round(1.0 - w_nn - w_xgb, 2)
            if w_lgb < 0 or w_lgb > 1:
                continue
            blend = w_xgb * xgb_val + w_lgb * lgb_val + w_nn * nn_val
            score = calculate_wmae(y_val, np.clip(blend, 0, None), is_holiday_val)
            if score < best_wmae:
                best_wmae, best_w = score, {"xgb": w_xgb, "lgb": w_lgb, "nn": w_nn}

    mlflow.log_params(best_w)
    mlflow.log_metric("WMAE", float(best_wmae))
    print("Best weights:", best_w)
    print(f"Best ensemble validation WMAE: {best_wmae:,.2f}")

Best weights: {'xgb': np.float64(0.0), 'lgb': np.float64(0.2), 'nn': np.float64(0.8)}
Best ensemble validation WMAE: 2,209.78
🏃 View run Ensemble_Weight_Search at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/10/runs/811b4f8778064747a16c9da6e4d766ff
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/10


## Notes

- All components and the blend are scored on the same validation rows with the shared
  `calculate_wmae`. No leakage.
- If the best blend beats N-BEATS alone (~2,277), the next step is to apply the same
  weights to the test predictions (`predictions/nn_test_preds.csv` + tree test preds)
  and submit.